# MiniMax H3 on Kaggle — Wan2GP + Turbo LoRA

Designed for **Kaggle T4**. This notebook keeps the large H3 model in `/tmp` to avoid filling `/kaggle/working`.

**Target test:** MiniMax H3 FL2VA Pruned 20B, 480p, ~5 seconds, Turbo EMA ckpt850 4 steps, 6 steps.


## 1. Check GPU and disk


In [ ]:
!nvidia-smi

import torch, os
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), 
          f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")

## 2. Clone Wan2GP


In [ ]:
%cd /kaggle/working
import os
if not os.path.exists('/kaggle/working/Wan2GP'):
    !git clone https://github.com/deepbeepmeep/Wan2GP.git
%cd /kaggle/working/Wan2GP


## 3. Install Wan2GP dependencies

In [ ]:
%cd /kaggle/working/Wan2GP

!pip install -r requirements.txt

## 4. Create temporary model

In [ ]:
!mkdir -p /tmp/minimax_h3
!df -h /tmp

## 5. Download the H3 pruned INT8 ConvRot base

This is about 21 GB, so it goes to `/tmp`.

In [ ]:
from huggingface_hub import hf_hub_download
import os

base = hf_hub_download(
    repo_id='DeepBeepMeep/MiniMax-H3',
    filename='MiniMax-H3-FL2VA-pruned_int8_convrot.safetensors',
    local_dir='/tmp/minimax_h3'
)
print('Base:', base)
!ls -lh /tmp/minimax_h3


## 6. Download the Qwen3-VL text encoder

Use the actual file discovered in the MiniMax-H3 repository.

In [5]:
encoder = hf_hub_download(
    repo_id='DeepBeepMeep/MiniMax-H3',
    filename='Qwen3-VL-32B-Instruct/Qwen3-VL-32B-Instruct-layer50_quanto_bf16_int8.safetensors',
    local_dir='/tmp/minimax_h3/Qwen3-VL-32B-Instruct'
)
print('Encoder:', encoder)
!du -sh /tmp/minimax_h3


## Optional speed profile — First Block Cache

This cell adds two selectable speed profiles using Wan2GP's existing MiniMax H3 cache. It does **not** enable caching automatically.

After the UI opens, select your usual **MiniMax H3 FL2VA Pruned** model and **Turbo EMA ckpt850** LoRA first. Then apply **Kaggle Speed - First Block Cache 0.06** from **Accelerators Profiles**. The profile keeps your model, LoRAs, steps, prompt, seed, resolution and frame count. It changes only the cache settings.

The cache can reuse transformer results when consecutive steps are similar. The lowest supported threshold (`0.06`) is used, with a 25% warmup. Output quality can change, and a four-step Turbo run may skip little or nothing; no speed increase is guaranteed.

To return to the previous calculation method, apply **Kaggle Speed - Cache OFF**, or set **Steps Skipping → Skip Steps Cache Type → None**. If this Wan2GP version does not support the feature, the cell skips installation and the original launch still works.

For a fair comparison, keep the same inputs, seed and step count; compare generation time after the models are already loaded. See [SPEED.md](SPEED.md) for the Persian guide.


In [ ]:
# Adds optional cache profiles only; the original launch and model setup stay intact.
import ast
import json
from pathlib import Path

wan_root = Path('/kaggle/working/Wan2GP')
handler_file = wan_root / 'models/minimax_h3/minimax_h3_handler.py'
cache_file = wan_root / 'models/minimax_h3/first_block_cache.py'
profile_dir = wan_root / 'profiles/minimax_h3_fl2va'

try:
    handler_tree = ast.parse(handler_file.read_text(encoding='utf-8'))
    supports_fbc = any(
        isinstance(node, ast.Dict) and any(
            isinstance(key, ast.Constant) and key.value == 'first_block_cache'
            and isinstance(value, ast.Constant) and value.value is True
            for key, value in zip(node.keys, node.values)
        )
        for node in ast.walk(handler_tree)
    )
    thresholds = next((
        ast.literal_eval(node.value)
        for node in ast.walk(handler_tree)
        if isinstance(node, ast.Assign) and any(
            isinstance(target, ast.Name) and target.id == 'FIRST_BLOCK_CACHE_THRESHOLDS'
            for target in node.targets
        )
    ), ())
    compatible = supports_fbc and 0.06 in thresholds and cache_file.is_file() and profile_dir.is_dir()
    if not compatible:
        print('Speed profiles skipped: this Wan2GP version has no compatible MiniMax H3 First Block Cache.')
    else:
        speed_profiles = {
            'Kaggle Speed - First Block Cache 0.06.json': {
                'skip_steps_cache_type': 'first_block',
                'skip_steps_multiplier': 0.06,
                'skip_steps_start_step_perc': 25,
                'help': 'First Block Cache enabled: threshold 0.06, warmup 25%. Other generation settings are kept. Speed and quality depend on the input.'
            },
            'Kaggle Speed - Cache OFF.json': {
                'skip_steps_cache_type': '',
                'help': 'Steps-skipping cache disabled. Other generation settings are kept.'
            }
        }
        for filename, settings in speed_profiles.items():
            (profile_dir / filename).write_text(json.dumps(settings, indent=4) + '\n', encoding='utf-8')
        print('Speed profiles ready. Apply "Kaggle Speed - First Block Cache 0.06" in Accelerators Profiles after selecting your model and Turbo LoRA.')
        print('Rollback: apply "Kaggle Speed - Cache OFF", or choose None in Steps Skipping.')
except (OSError, SyntaxError, ValueError, TypeError) as error:
    print(f'Speed profiles skipped ({type(error).__name__}). Continue with the original launch cell.')


## 7. Launch Wan2GP

The current Wan2GP build exposes `--lora-dir-minimax-h3`, so the LoRA can stay in `/tmp` instead of consuming working-disk space.

**Run this cell last.** It will keep the notebook cell occupied while Wan2GP is running. Open the Gradio URL printed by the cell.

In [ ]:
%cd /kaggle/working/Wan2GP

!python wgp.py --share

## 7. URLs

For your MiniMax H3 setup:

### Main Checkpoints

```text
/tmp/minimax_h3/MiniMax-H3-FL2VA-pruned_int8_convrot.safetensors
```

### Text Encoder Checkpoints

```text
/tmp/minimax_h3/Qwen3-VL-32B-Instruct/Qwen3-VL-32B-Instruct/Qwen3-VL-32B-Instruct-layer50_quanto_bf16_int8.safetensors
```

### Video VAE File

Keep the existing MiniMax H3 **Video VAE** entry from the source model.

### Audio VAE File

Keep the existing MiniMax H3 **Audio VAE** entry from the source model.



## 8. Suggested first-generation settings

- Model: **MiniMax H3 / FL2VA Pruned 20B**
- Turbo LoRA: **Turbo EMA ckpt850 4 steps**
- LoRA strength: **1.0**
- Steps: **4**
- Scheduler: **simple** if exposed by the H3 Turbo workflow
- Resolution: **480p**
- Duration: **~5 seconds / 124 frames**
- Keep other acceleration methods OFF for the first test.

## 9. GPU monitoring (run in a separate cell after starting generation)


In [ ]:
!nvidia-smi